In [ ]:
import itertools
import os
import warnings

from typing import Union

import hictkpy as htk
import matplotlib.colors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats
import seaborn as sns



warnings.simplefilter(action = 'ignore')
np.seterr(divide = 'ignore')

## Hi-Res FISH

In [4]:
def Euclidean_Dist(df1, df2, cols=['x','y','z']):

    'Calculates three-dimentional euclidian distance between i(x,y,z) and j(x,y,z)'
    
    return np.linalg.norm(df1[cols].values - df2[cols].values,
                   axis=1)

In [ ]:
#open files with coordinates of center of masses for each channel obtained from FiJi

farred=pd.read_csv(r'/Users/anastasiastompel/Desktop/sec_repl_hiresfish/prk_farred.csv')
red=pd.read_csv(r'/Users/anastasiastompel/Desktop/sec_repl_hiresfish/prk_red.csv')
green=pd.read_csv(r'/Users/anastasiastompel/Desktop/sec_repl_hiresfish/prk_green.csv')

#calculate pair-wise distances

farred_red_dist=Euclidean_Dist(farred, red)
farred_green_dist=Euclidean_Dist(farred, green)
red_green_dist=Euclidean_Dist(red, green)

#combine in one dataframe and save .csv
dt={'KRT5_LCR2':farred_red_dist, 'KRT5_LCR1':farred_green_dist, 'LCRs':red_green_dist}
df_krt_dist = pd.DataFrame(dt)

#### Pair-wise distances distribution

In [ ]:
#open .csv file with data including pair-wise distances for all cell lines and all combinations of i and j

dists = pd.read_csv(r'/Users/anastasiastompel/Desktop/kerat data/FISH_DATA_SEC_REPL/sum_dist_sec_repl_suppl.csv')

#do not include distances >900 nm, possible artifacts or signals from separate loci
dists_thresholded = dists.clip(0, 0.9)
plt.figure(figsize=(9,6))
sns.set_context("poster")

pal = ['#c95977', '#940000', '#59c977', '#53b6ee', '#3f94e4', '#4753d6']

sns.set(style="white")
sns.boxplot(data=dists_thresholded, palette=pal, linewidth=2, notch = True)
#sns.stripplot(data=dists_thresholded, color = 'black', alpha = 0.5)

plt.xticks(fontsize=12, rotation = 45, ha="right", rotation_mode="anchor")
plt.yticks(fontsize=12)
plt.title('Pair-wise distances', fontsize=20, pad=10)
plt.ylabel('Distance, mkm', fontsize=15)
plt.ylim(0, 1)
plt.axvline(5.5,color = 'black',linestyle = '--', linewidth = 2)
plt.axvline(11.5,color = 'black',linestyle = '--', linewidth = 2)

In [ ]:
combs = set(itertools.combinations(dists_thresholded.columns, 2))
for i in list(combs):
    print(f'Mann-Whitney {i[0]} and {i[1]} is', scipy.stats.mannwhitneyu(x=dists_thresholded[i[0]].dropna(), y=dists_thresholded[i[1]].dropna()))

#### Percentage of triple complexes

In [ ]:
plt.figure(figsize=(4,7))

def count_proportion(vec):
    return (vec[vec <= 0.25].count())/vec.count()*100


proportion = dists_thresholded.apply(count_proportion, axis=0)

pal = ['#940000', '#53b6ee', '#3f94e4', '#4753d6']

sns.barplot(data = proportion, palette=pal)
plt.xticks(fontsize=10, rotation = 45, ha="right", rotation_mode="anchor")
plt.ylim(0, 50)

#### Exact Fisher's test

In [ ]:
dists = (
    pd.read_csv('/Users/anastasiastompel/Desktop/kerat data/FISH_DATA_SEC_REPL/sum_dist_sec_repl_suppl.csv')
      .clip(0, 0.9)
      .drop(columns=['KRT5_LCR2_HaCaT', 'KRT5_LCR1_HaCaT', 'LCRs_HaCaT'])
)

groups = ['prk', 'fb', 'ipsc0', 'ipsc7', 'ipsc21']
markers = ['KRT5_LCR2', 'KRT5_LCR1', 'LCRs']

rows = []
for g in groups:
    sub = dists[[f'{m}_{g}' for m in markers]]
    pos = sub.le(0.25).all(axis=1).sum()
    total = sub.notna().all(axis=1).sum()
    rows.append({'group': g, 'positive': pos, 'non': total - pos})

summary = pd.DataFrame(rows).set_index('group')

In [ ]:
ipsc21 = summary.loc['ipsc21'].to_list()
second = summary.loc['fb'].to_list()

for_fisher = []
for_fisher.append(ipsc21)
for_fisher.append(second)

odd_ratio, p_value = scipy.stats.fisher_exact(for_fisher)
print('odd ratio is : ' + str(odd_ratio))
print('p_value is : ' + str(p_value))

## BAC FISH

In [ ]:
df_rg=pd.read_csv(r'/Users/anastasiastompel/Desktop/sec_repl_hiresfish/RgBACFISH.csv')

plt.figure(figsize=(3,5))
pal = ['#c95977', '#940000', '#59c977', '#53b6ee', '#3f94e4', '#4753d6']
sns.boxplot(data=df_rg, notch=True)
plt.ylabel('Radius of gyration, um^2', fontsize=10)
plt.ylim(0.1, 0.7)
plt.xticks(fontsize=8, rotation = 45, ha="right", rotation_mode="anchor")

combs = set(itertools.combinations(df_rg.columns, 2))
for i in list(combs):
    print(f'Mann-Whitney {i[0]} and {i[1]} is', scipy.stats.mannwhitneyu(x=df_rg[i[0]].dropna(), y=df_rg[i[1]].dropna()))

## RNA-seq

In [ ]:
df = pd.read_csv('/Users/anastasiastompel/Desktop/simple_counts_ipsc_hac_pk.txt', sep='\t')
df2 = pd.read_csv('/Users/anastasiastompel/Desktop/Genes_2.txt', sep='\t', header = None, names=["chr", 'start', 'end', 'Geneid'])
df2['Length'] = df2['end'] - df2['start']
df3 = df2[['Geneid', 'Length']]
genes_krt_locus = pd.merge(df, df3, on="Geneid", how="inner")
genes_krt_locus['pk.bam'] = genes_krt_locus[['d147A.bam', 'pr_rna.bam']].mean(axis=1)
genes_krt_locus_fin = genes_krt_locus[['Geneid', '0_day_rna.bam', '7_day_rna.bam', '21_day_rna.bam', 'pk.bam', 'HaCaT.bam', 'Length']]

Formula
RPKM =   numReads / ( geneLength/1000 * totalNumReads/1,000,000 )

numReads - number of reads mapped to a gene sequence
geneLength - length of the gene sequence
totalNumReads - total number of mapped reads of a sample

In [ ]:
gene_col = 'Geneid'
length_col = 'Length'
sample_cols = [c for c in genes_krt_locus_fin.columns if c.endswith('.bam')]

# копируем только нужное
counts = genes_krt_locus[[gene_col, length_col] + sample_cols].copy()

library_sizes = counts[sample_cols].sum()

# RPKM = counts * 1e9 / (gene_length * total_mapped_reads)
rpkm = counts.copy()
for sample in sample_cols:
    rpkm[sample] = counts[sample] / ((counts[length_col]/1000) * (library_sizes[sample]/1000000))

#rpkm.to_csv('/Users/anastasiastompel/Desktop/rpkm_all_norm.csv')

In [ ]:
rpkm_genes = pd.read_csv('/Users/anastasiastompel/Desktop/rna_seq_final.csv', sep=',')
rpkm_genes.set_index(rpkm_genes.columns[0], inplace=True)
plt.figure(figsize=(10, 0.4))

genes = np.log10(rpkm_genes)
genes.replace([np.inf, -np.inf], 0, inplace=True) 
except_k18_8 = genes.iloc[:-2,:]
sns.heatmap(data = except_k18_8.T, linewidths=0.5, linecolor='black', cmap='RdYlBu_r',cbar_kws={'shrink': 1, 'pad': 0.1, 'aspect': 10})
sns.set(font="Arial")
plt.subplots_adjust(left=0.1, right=0.9, bottom=0.1, top=0.9)

## ChIP-seq

### H3K27 acetyl FACS keratinocytes

In [ ]:
def assign_region(row):
    s, e = row['start1'], row['end1']
    if s <= 52159000 and e >= 52143000:
        return 'KSE1'
    elif s <= 52881000 and e >= 52855000:
        return 'KSE2'
    elif s <= 52590000 and e >= 52565000:
        return 'PE1'
    elif s <= 52685000 and e >= 52670000:
        return 'KRT1'
    elif s <= 52530000 and e >= 52514000:
        return 'KRT5'
    elif s < 52143000:
        return 'outside'
    else:
        return 'other'
 
    
def integrative_score(data, name):
    data['K27_length'] = data['end1'] - data['start1']
    krt_k27_score = data[['start1', 'end1', 'K27_score', 'K27_length', 'region_name']]
    df_unique = krt_k27_score.drop_duplicates()
    df_unique['integral_intensity'] = df_unique['K27_score'] * df_unique['K27_length']
    atac_counts = data.groupby(['start1', 'end1']).size().reset_index(name='ATAC_count')
    df_unique = df_unique.merge(atac_counts, on=['start1', 'end1'], how='left')
    df_unique['ATAC_count'] = df_unique['ATAC_count'].fillna(0).astype(int)
    filtered_df = df_unique[df_unique['region_name'] != 'outside']
    sns.stripplot(data=filtered_df, x='start1', y='integral_intensity', hue='region_name', s=5)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0)
    plt.xticks(np.arange(0, len(df_unique['start1'])+1, 7))
    plt.title(f'{name} integrative K27 score')
    plt.axhline(20000,color = 'grey',linestyle = '--', linewidth = 1)
    return filtered_df


def mm_to_pt(mm):
    return mm * 72 / 25.4

In [ ]:
bedtools_inter_res1 = pd.read_csv('/Users/anastasiastompel/Desktop/KeratinsArticleMAIN/CTCF_annot_diff/Ker1_for_scatter_new_gap600.bed', 
                                 sep='\t', header = None, names=["chr1", 'start1', 'end1', 'K27_score', 
                                                                 "chr2", 'start2', 'end2', 'ATAC_score'])

bedtools_inter_res1['region_name'] = bedtools_inter_res1.apply(assign_region, axis=1)

filtered_df = bedtools_inter_res1[bedtools_inter_res1['region_name'] != 'outside']
df_k1 = integrative_score(bedtools_inter_res1, 'KRT1')
df_k1['chrom'] = 'chr12'
plt.rcParams["figure.figsize"] = 6, 5
pal = ['gray','#FF6666', '#3771c8', '#BC0E0E']
sns.stripplot(data=df_k1, x='ATAC_count', y='integral_intensity', hue='region_name', s=10,
              edgecolor='black', linewidth=0.709, palette=pal)
plt.title('KRT1')
plt.legend('',frameon=False)

### CTCF iPSC


In [ ]:
def assign_region_ctcf(row):
    s, e = row['start'], row['end']
    if s <= 52158000 and e >= 52143000:
        return 'KSE1'
    elif s <= 52880000 and e >= 52858000:
        return 'KSE2'
    elif s < 52880000 and e > 52143000:
        return 'KSE1-KSE2'
    else:
        return 'outside'

In [ ]:
names = ['chr', 'ips7_re1', 'ips7_re2', 'ips21_re1', 'ips21_re2']
df_ipsc_ctcf = pd.read_csv('/Users/anastasiastompel/Desktop/KeratinsArticleMAIN/CTCF_annot_diff/ctcf/new_ctcf_0_7/norm-matrix-deseq2_ipsc_ctcf_7_21.txt', 
                           sep='\t', header=None, names=names)
df_ipsc_ctcf[["chrom", 'start', 'end']] = df_ipsc_ctcf["chr"].str.split("_", n=2, expand=True)
df_ipsc_ctcf_edit = df_ipsc_ctcf[["chrom", 'start', 'end', 'ips7_re1', 'ips7_re2', 
                                  'ips21_re1', 'ips21_re2']]
df_ipsc_ctcf_edit = df_ipsc_ctcf_edit.astype({'start': int, 'end': int})
df_ipsc_ctcf_edit['region_name'] = df_ipsc_ctcf_edit.apply(assign_region_ctcf, axis=1)

data_ipsc_ctcf_7_vs_21 = pd.read_csv('/Users/anastasiastompel/Desktop/KeratinsArticleMAIN/CTCF_annot_diff/ctcf/new_ctcf_0_7/result_ipsc_ctcf_7_21.txt', 
                                    sep='\t')
data_fc_ipsc_ctcf_7_vs_21 = data_ipsc_ctcf_7_vs_21[['id', 'log2FoldChange', 'pvalue']]
data_fc_ipsc_ctcf_7_vs_21[["chrom", 'start', 'end']] = data_fc_ipsc_ctcf_7_vs_21["id"].str.split("_", n=2, expand=True)
df_fc_ipsc_21vs7_edit = data_fc_ipsc_ctcf_7_vs_21[["chrom", 'start', 'end', 'log2FoldChange', 'pvalue']]
df_fc_ipsc_21vs7_edit = df_fc_ipsc_21vs7_edit.astype({'start': int, 'end': int})
df_sorted = df_fc_ipsc_21vs7_edit.sort_values('start')
df_ctrl_ctcf = df_ipsc_ctcf_edit[["chrom", 'start', 'end', 'ips7_re1', 'ips7_re2', 'region_name']]
merged_df = pd.merge(df_sorted, df_ctrl_ctcf, on=['chrom', 'start', 'end'], how='inner') 
merged_df_selected = merged_df.query("region_name != 'outside'")

In [ ]:
p_val = 0.0001
non_sign = merged_df_selected[(merged_df_selected['pvalue'] >= p_val)]
sign = merged_df_selected[(merged_df_selected['pvalue'] < p_val)]
markers_dict = {'KSE1-KSE2': 'o', 'KSE1': '^', 'KSE2': 'v'}

sns.scatterplot(data=non_sign, x='ips7_re1', y='log2FoldChange', color="lightgray", label='non significant', 
                edgecolor='black', linewidth=0.709, s=50, legend=False)

sign_up = sign[sign['log2FoldChange'] > 0]
sign_down = sign[sign['log2FoldChange'] < 0]

sns.scatterplot(data=sign_up, x='ips7_re1', y='log2FoldChange', color='#c92437', style='region_name', 
                edgecolor='black', linewidth=0.709, s=50, markers=markers_dict, legend=False)

sns.scatterplot(data=sign_down, x='ips7_re1', y='log2FoldChange', color='#3771c8', style='region_name', 
                edgecolor='black', linewidth=0.709, s=50, markers=markers_dict, legend=False)

plt.axhline(0,color = 'grey',linestyle = '--', linewidth = 1)
plt.title('ipsc7_ipsc21_CTCF')
plt.ylim(-3,3)
plt.xlim(0, 4000)
plt.text(x = 1800, y = -2.5, s = 'p-value = 10**-4', size = 9, color = 'black')

In [ ]:
merged_df_selected_wo_kse = merged_df_selected.query("region_name == 'KSE1-KSE2'")
total_count = len(merged_df_selected_wo_kse)
non_sign = merged_df_selected_wo_kse[(merged_df_selected_wo_kse['pvalue'] >= p_val)]
sign = merged_df_selected_wo_kse[(merged_df_selected_wo_kse['pvalue'] < p_val)]
non_significant_count = len(non_sign)
significant_count = len(sign)
up = len(sign[sign['log2FoldChange'] > 0])
down = len(sign[sign['log2FoldChange'] < 0]) 
print(f'Total {total_count}, Non_sign peaks count: {non_significant_count}, sign peaks count: {significant_count}')
print(f'Among sign: {up} is up, {down} is down')

## C-TALE

### Boxplots

In [ ]:
def convert_coords_to_bins(region_coords: str, resolution: int, region_for_boxplot: dict) -> dict:

        "Convert genomic coordinates to number of bins"

        bins_dict = {}
        first_bin_of_matrix = int(region_coords.split(':')[1].split('-')[0])
        last_bin_of_matrix = int(region_coords.split(':')[1].split('-')[1])
        for key, value in region_for_boxplot.items():
                if value[2] > value[0] and value[3] > value[1]:
                        bins = [int((x-first_bin_of_matrix)/resolution) if type(x) is int and (last_bin_of_matrix >= x >= first_bin_of_matrix) else print(f'{x} is not integer or out of range!') for x in value] 
                        bins_dict[key] = bins
        return bins_dict


def get_boxplot_values(path: list, resolution: int, region_coords: str, region_for_boxplot: dict) -> pd.DataFrame:

        "Obtain matrix values for ROI."

        bins_for_boxplot = convert_coords_to_bins(region, res, region_for_boxplot)
        
        merged = None     
        for cool in path:
                colname = os.path.splitext(os.path.basename(cool))[0][:15]
                file = htk.File(cool, resolution)
                sel = file.fetch(region_coords, join=True, normalization='weight')
                matrix = sel.to_numpy()
                for key, value in bins_for_boxplot.items():
                        selected_values = matrix[value[0]:value[2], value[1]:value[3]].flatten()
                        selected_values = selected_values[selected_values!=0]
                        selected_values = pd.DataFrame({key+ '_' + colname: selected_values})
                        if merged is None:
                                merged = pd.DataFrame()
                                merged = selected_values
                        else:
                                merged = pd.concat([merged, selected_values], axis=1)
        return merged

In [ ]:
region_for_boxplot = {'KSE1-KSE2': [52_142_000, 52_866_000, 52_160_000, 52_884_000]}

cools_path = ['/Users/anastasiastompel/Desktop/kerat data/ammonium/Amm_ctrl_L1_1_1000_hg38_gen_coord.mcool'] # path to mcool files

res = 2000 # resolution in bp
region = '12:51945446-52971582' # UCSC-like, this coords for keratin locus 12q13.13

data = get_boxplot_values(cools_path, res, region, region_for_boxplot)

#plt.rcParams["figure.figsize"] = 3, 4
my_colors = ['#708090', '#A145F2', '#062db7', '#3139c2', '#2e7ac7','#43a1ce', '#57c7d6'] #set palette
sns.set_palette(my_colors)

plt.rcParams["figure.figsize"] = 22, 7


ax = sns.boxplot(data=data, notch=False, linewidth=3, showfliers=True, boxprops={"edgecolor": "black", "linewidth": 2.5},
            whiskerprops={"color": "black", "linewidth": 2},
            capprops={"color": "black", "linewidth": 2},
              medianprops={"color": "black", "linewidth": 3})


plt.ylabel('Relative ligation frequency', fontsize=18)
plt.xticks(rotation = 45)
plt.title('KSE1-KSE2', fontsize=20)
plt.ylim(0,0.035)
plt.yticks(fontsize=15)


### Virtual 4C

In [ ]:
def check_coords(x: list,y: list) -> list:

    "Check is list of coordinates corrects (end > start)"

    res = all(b > a for a, b in zip(x, y))
    return res

def obtain_bin_coords(region_coords: str, resolution:int, start: Union[list, int], end: Union[list, int]) -> Union[list, int]:

    "Converts genomic coordinates to number of bins, check are coordinates correct. Function can works with individual integer or list of integer."

    first_bin_of_matrix = int(region_coords.split(':')[1].split('-')[0])
    last_bin_of_matrix = int(region_coords.split(':')[1].split('-')[1])
    if type(start) and type(end) is list:
        if len(start) == len(end) and check_coords(start, end):
            start_bins = [int((x-first_bin_of_matrix)/resolution) if type(x) is int and (last_bin_of_matrix >= x >= first_bin_of_matrix) else print(f'{x} is not integer or out of range!') for x in start] 
            end_bins = [int((y-first_bin_of_matrix)/resolution) if type(y) is int and (last_bin_of_matrix >= y >= first_bin_of_matrix) else print(f'{y}  is not integer or out of range!') for y in end]
            return start_bins, end_bins, first_bin_of_matrix
        return ("Number of start bins and end bins is not equal!")
    elif (type(start) and type(end) is int) and end > start:
        if (last_bin_of_matrix >= start >= first_bin_of_matrix) and (last_bin_of_matrix >= end >= first_bin_of_matrix):
            return int((start - first_bin_of_matrix)/resolution), int((end - first_bin_of_matrix)/resolution), first_bin_of_matrix
        return "Start or end coordinates is out of range."
    return ('Wrong input type! List of integer or integer is needed.')

def get_values(path: list, resolution: int, region_coords: str, start: Union[list, int], end: Union[list, int]) -> pd.DataFrame:

    "Get pandas Dataframe of matrix values for region of interest. Calculate mean value of several strings of 2D matrix."

    merged = {}
    list_of_start, list_of_end, first = obtain_bin_coords(region_coords, resolution, start, end)
    for cool in path:
        colname = os.path.splitext(os.path.basename(cool))[0]
        file = htk.File(cool, resolution)
        sel = file.fetch(region_coords, join=True, normalization='weight')
        matrix = sel.to_numpy()
        if type(list_of_start) and type(list_of_end) is list:
            for start_bin, end_bin in zip(list_of_start, list_of_end):
                viewpoint = matrix[start_bin:end_bin,:]
                viewpoint = np.mean(viewpoint, axis = 0)
                viewpoint = np.nan_to_num(viewpoint)
                merged[colname + f'_{start_bin * resolution + first} - {end_bin * resolution + first}'] = viewpoint
        else:
            viewpoint = matrix[(list_of_start):list_of_end,:]
            viewpoint = np.mean(viewpoint, axis = 0)
            viewpoint = np.nan_to_num(viewpoint)
            merged[colname + f'_{list_of_start * resolution + first} - {list_of_end * resolution + first}'] = viewpoint
    values_dataframe = pd.DataFrame(data = merged)
    return values_dataframe

In [ ]:
cools_path = ['/Users/anastasiastompel/Desktop/Новая папка/Amm_ctrl_L1_1_1000_hg38_gen_coord.mcool'] # path to mcool files

res = 4000 # resolution in bp
region = '12:51945446-52971582' # UCSC-like, this coords for keratin locus 12q13.13
start = [52520255, 52143128] # list of int or int
end = [52526594, 52158933]


dd = get_values(cools_path, res, region, start, end)

plt.rcParams["figure.figsize"] = 10, 5
for column in dd.columns:
    plt.plot(dd[column], label=f'{column}')
    plt.ylim(0, 0.08)
plt.legend()
None

### Substraction matrices

In [ ]:
def substract_like_juicer(ctrl, treat):
    ctrl_av_val = np.nanmean(ctrl)
    aux_av_val = np.nanmean(treat)

    const = (ctrl_av_val/2 + aux_av_val/2)

    ctrl_norm_mtx = ctrl/ctrl_av_val
    aux_norm_mtx = treat/aux_av_val
    diff_matrix = (aux_norm_mtx - ctrl_norm_mtx) * const
    return diff_matrix


def create_df(treatments, coord):
    data_subs = pd.DataFrame()
    for treat in treatments.items():
        choosen_bins = treat[1][coord[0]: coord[1], coord[2]:coord[3]].flatten()
        choosen_bins = choosen_bins[choosen_bins!=0]
        choosen_bins_df = pd.DataFrame({treat[0]: choosen_bins})
        if data_subs.empty:
            data_subs = choosen_bins_df
        else:
            data_subs = pd.concat([data_subs, choosen_bins_df], axis=1)
    return data_subs

In [ ]:
lcr1_krt5 = [285,291,99,106]
lcr2_krt5 = [285,291,459,468]
lcr1_lcr2 = [99,106,459,468]
pe1_krt5 = [285,291,314,319]
pe1_lcr1 = [99,106,314,319]
pe1_lcr2 = [459,468,314,319]

cbs356_cbs193 = [107,113, 188,193]

cbs356_cbs307 = [107,113, 131,137] #loop1
cbs129_cbs13 = [221,227,279,285] #loop2
cbs363_cbs140 = [104,110,215,221]

In [ ]:
#open maps
ctrl = htk.File('/Users/anastasiastompel/Desktop/kerat data/cool_maps/drug_treat_kerat/downsampled/dmso_down_gen_coord_4kb.hic', 2000)
sel_c = ctrl.fetch('12:51945446-52971582', join=True, normalization='KR')
m_ctrl = sel_c.to_numpy()

flv = htk.File('/Users/anastasiastompel/Desktop/kerat data/cool_maps/drug_treat_kerat/downsampled/flv_gen_coord_4kb.hic', 2000)
sel_f = flv.fetch('12:51945446-52971582', join=True, normalization='KR')
m_flv = sel_f.to_numpy()

trp = htk.File('/Users/anastasiastompel/Desktop/kerat data/cool_maps/drug_treat_kerat/downsampled/trp_down_gen_coord_4kb.hic', 2000)
sel_t = trp.fetch('12:51945446-52971582', join=True, normalization='KR')
m_trp = sel_t.to_numpy()

jq1 = htk.File('/Users/anastasiastompel/Desktop/kerat data/cool_maps/drug_treat_kerat/downsampled/jq_down_gen_coord_4kb.hic', 2000)
sel_j = jq1.fetch('12:51945446-52971582', join=True, normalization='KR')
m_jq1 = sel_j.to_numpy()



ctrl_hex = htk.File('/Users/anastasiastompel/Desktop/kerat data/cool_maps/hexandiol_kerat/hex_tween_permeab_kerat_gen_coord.hic', 2000)
sel_ch = ctrl_hex.fetch('12:51945446-52971582', join=True, normalization='KR')
m_ctrl_hex = sel_ch.to_numpy()

hexan = htk.File('/Users/anastasiastompel/Desktop/kerat data/cool_maps/hexandiol_kerat/hex_kerat_gen_coord.hic', 2000)
sel_h = hexan.fetch('12:51945446-52971582', join=True, normalization='KR')
m_hex = sel_h.to_numpy()

flv_sub = substract_like_juicer(m_ctrl, m_flv)
trp_sub = substract_like_juicer(m_ctrl, m_trp)
jq1_sub = substract_like_juicer(m_ctrl, m_jq1)
hex_sub = substract_like_juicer(m_ctrl_hex, m_hex)

treatments = {'FLV': flv_sub, 'TRP': trp_sub, 'JQ1': jq1_sub, 'HEX': hex_sub}

In [ ]:
plt.rcParams["figure.figsize"] = 3, 3
colors = ("white","#F5A623","#D0021B", "black")
cmap = matplotlib.colors.LinearSegmentedColormap.from_list("", colors)

plt.matshow(flv_sub[1:59,36:51], cmap='bwr', vmin=-60, vmax=60) # visualize map
plt.axis('off')

#plt.matshow(np.log10(m_aux), cmap=cmap, vmin=0.5, vmax=3.8)
None

In [ ]:
dd = create_df(treatments, cbs363_cbs140)
plt.rcParams["figure.figsize"] = 0.5, 5
#pal = ['#c95977', '#3f94e4', 'violet', 'darkgreen']

sns.boxplot(data=dd, notch=True, linewidth=2, showfliers=False, palette=pal, linecolor='black')

#plt.ylabel('Relative ligation frequency', fontsize=12)
plt.ylim(-15, 15)
#plt.axhline(0,color = 'black',linestyle = '--', linewidth = 2)
plt.title('cbs363_cbs140')

In [ ]:
p_value = scipy.stats.wilcoxon(dd).pvalue